In [43]:
import pandas as pd
from tqdm import tqdm

## Analyze Pfam based on all parameters

In [47]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','col0']
runs = ["","_cat","_no_co"]
for run in runs:
    pfam_nlrs=pd.DataFrame()
    pfam_dom_gene=pd.DataFrame()
    pfam_dom_iso=pd.DataFrame()
    pfam_orf_gene=pd.DataFrame()
    pfam_orf_iso=pd.DataFrame()
    for acc in tqdm(accessions):
        # data preparation
        pfam = pd.read_csv(f"output/n5_genes_/{acc}{run}/{acc}{run}_nlr_like.prot.fasta.tsv", sep="\t", names=["Query protein","Sequence MD5 digest","Sequence length","Analysis","Signature accession","Signature description","Start location","Stop location","Score (e-value)","Status (T: true)","Date","InterPro annotations - accession","InterPro annotations - description","GOTerms"])
        pfam = pfam.loc[pfam['Analysis']=="Pfam"]
        pfam["gene"]=pfam["Query protein"].str.split(".").str[0]
        pfam["isoform"]=pfam["Query protein"].str.split(".").str[0]+"."+pfam["Query protein"].str.split(".").str[1]

        # n query proteins
        # n query isoforms
        # n query genes

        ## NB-ARC analysis
        # get all transdecoder orfs with NB-ARC relation
        nbarc_trans = pfam.loc[pfam["Signature description"]=="NB-ARC domain"]["Query protein"].tolist()
        # extract full info from Pfam
        nbarc = pfam.loc[pfam["Query protein"].isin(nbarc_trans)]
        # NB-ARC gene:
        nlr_genes_dict={"gene":[],"TIR":[],"LRR":[],"CC":[]}
        #print(nlr_genes_dict)
        for i in nbarc.groupby("gene"):
            # get gene id
            nlr_genes_dict["gene"].append(i[0])

            # get TIR
            if "TIR domain" in i[1]["Signature description"].tolist():
                nlr_genes_dict["TIR"].append(1)
            else:
                nlr_genes_dict["TIR"].append(0)

            # get LRR
            lrr_signature=i[1]["Signature description"]
            lrr=0
            for j in lrr_signature:
                if "Leucine" in j:
                    lrr+=1
                    break
            nlr_genes_dict["LRR"].append(lrr)

            # get CC (RX domain in pfam!)

            cc_signature=i[1]["Signature description"]
            cc=0
            for j in cc_signature:
                if "Rx" in j:
                    cc+=1
                    break
            nlr_genes_dict["CC"].append(cc)
        nlr_out=pd.DataFrame.from_dict(nlr_genes_dict, orient="index").transpose()
        nlr_out["accession"]=acc
        pfam_nlrs=pd.concat([pfam_nlrs, nlr_out])


        ## Pfam analysis
        # gene
        dom_gene=pfam[["gene","InterPro annotations - description"]].drop_duplicates()
        dom_gene=dom_gene["InterPro annotations - description"].value_counts()
        dom_gene.rename(acc, inplace=True)
        pfam_dom_gene=pd.concat([pfam_dom_gene, dom_gene.head(n=11)], axis=1, sort=False)

        # isoform
        dom_iso=pfam[["isoform","InterPro annotations - description"]].drop_duplicates()
        dom_iso=dom_iso["InterPro annotations - description"].value_counts()
        dom_iso.rename(acc, inplace=True)
        pfam_dom_iso=pd.concat([pfam_dom_iso, dom_iso.head(n=11)], axis=1, sort=False)

        ## ORF analysis
        # gene
        orf_gene=pfam[["gene","GOTerms"]].drop_duplicates()
        orf_gene["GOTerms_list"]=orf_gene["GOTerms"].str.split("|")
        orf_gene=orf_gene.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_gene.rename(acc, inplace=True)
        pfam_orf_gene=pd.concat([pfam_orf_gene, orf_gene.head(n=11)], axis=1, sort=False)

        # isoform
        orf_iso=pfam[["isoform","GOTerms"]].drop_duplicates()
        orf_iso["GOTerms_list"]=orf_iso["GOTerms"].str.split("|")
        orf_iso=orf_iso.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_iso.rename(acc, inplace=True)
        pfam_orf_iso=pd.concat([pfam_orf_iso, orf_iso.head(n=11)], axis=1, sort=False)
    pfam_nlrs.to_csv(f"output/n5_genes/pfam{run}_nlrs.tsv", sep="\t")
    pfam_dom_gene.to_csv(f"output/n5_genes/pfam{run}_dom_gene.tsv", sep="\t")
    pfam_dom_iso.to_csv(f"output/n5_genes/pfam{run}_dom_iso.tsv", sep="\t")
    pfam_orf_gene.to_csv(f"output/n5_genes/pfam{run}_orf_gene.tsv", sep="\t")
    pfam_orf_iso.to_csv(f"output/n5_genes/pfam{run}_orf_iso.tsv", sep="\t")

  0%|                                                                                                                                                                                                                 | 0/18 [00:00<?, ?it/s]

at6923


 11%|██████████████████████▎                                                                                                                                                                                  | 2/18 [00:00<00:03,  4.11it/s]

at6929
at7143


 22%|████████████████████████████████████████████▋                                                                                                                                                            | 4/18 [00:01<00:05,  2.68it/s]

...

at9852
yes
at9879


 89%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                      | 16/18 [00:04<00:00,  4.01it/s]

at9883
yes
at9900


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:04<00:00,  4.08it/s]

yes
col0


### Print NLR like genes

In [70]:
nlr_domains=pd.read_csv("output/n5_genes/pfam_no_co_nlrs.tsv", sep="\t")#.set_index("Unnamed: 0")

In [77]:
nlr_domains.drop("Unnamed: 0", axis=1).stack()

0   gene         G15275
    TIR               1
    LRR               1
    CC                0
    accession    at6923
                  ...  
56  gene         G14719
    TIR               0
    LRR               1
    CC                0
    accession      col0
Length: 285, dtype: object

### Get most dominant GOs

In [66]:
GO_gene=pd.read_csv("output/n5_genes/pfam_no_co_orf_gene.tsv", sep="\t").set_index("Unnamed: 0")
GO_gene["sum"]=GO_gene.sum(axis=1)
GO_gene_sorted=GO_gene.drop("-").sort_values("sum", ascending=False)
for i, j in zip(GO_gene_sorted.index, GO_gene_sorted["sum"].tolist()):
    print(i+"\t"+str(j)+"\n")

GO:0005515	1418.0

GO:0005524	1265.0

GO:0006468	939.0

GO:0004672	939.0

GO:0016020	618.0

GO:0008270	450.0

GO:0003677	341.0

GO:0006355	228.0

GO:0055085	217.0

GO:0003676	192.0

GO:0003723	104.0

GO:0005634	24.0

GO:0016887	21.0

GO:0008017	12.0

GO:0046872	12.0



### Get most dominant Pfams

In [64]:
pfam_gene=pd.read_csv("output/n5_genes/pfam_no_co_dom_gene.tsv", sep="\t").set_index("Unnamed: 0")
pfam_gene["sum"]=pfam_gene.sum(axis=1)
pfam_gene_sorted=pfam_gene.drop("-").sort_values("sum", ascending=False)
for i, j in zip(pfam_gene_sorted.index, pfam_gene_sorted["sum"].tolist()):
    print(i+"\t"+str(j)+"\n")

Pentatricopeptide repeat	809.0

Protein kinase domain	597.0

Leucine-rich repeat	468.0

Serine-threonine/tyrosine-protein kinase, catalytic domain	420.0

E motif	361.0

Leucine-rich repeat-containing N-terminal, plant-type	356.0

Helicase, C-terminal	198.0

DYW domain	162.0

ABC transporter-like, ATP-binding domain	127.0

WD40 repeat	111.0

Bulb-type lectin domain	57.0

ATPase, AAA-type, core	46.0

DEAD/DEAH box helicase domain	11.0

S-locus glycoprotein domain	11.0

Kinesin motor domain	10.0

Ankyrin repeat	10.0

SANT/Myb domain	10.0

ABC transporter type 1, transmembrane domain	9.0

MULE transposase domain	9.0

F-box domain	8.0

RNA recognition motif domain	7.0

